<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/finance_agentic_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.8/782.8 MB 66.7 MB/s  0:00:13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [2]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


GPU Offload Supported: True


In [1]:
import os
import gc
import time
import subprocess
import shutil
import hashlib
from huggingface_hub import hf_hub_download, list_repo_files
from llama_cpp import Llama

class FinancialPentadCluster:
    def __init__(self,
                 model_configs: dict,
                 inference_seed: int = 123):
        self.inference_seed = inference_seed
        self.model_configs = model_configs
        self.agents = {}
        self._verify_and_cache_paths()

    def _robust_download(self, repo_id: str, filename: str, max_retries: int = 5) -> str:
        delay = 10
        for attempt in range(1, max_retries + 1):
            try:
                print(f"[Hugging Face Hub] Downloading '{filename}' from {repo_id} (Attempt {attempt}/{max_retries})...")
                return hf_hub_download(
                    repo_id=repo_id,
                    filename=filename,
                    force_download=False
                )
            except Exception as e:
                print(f"[Warning] Download encountered an error: {e}")
                if attempt == max_retries:
                    raise RuntimeError(f"Failed to download {filename} from {repo_id} after {max_retries} attempts.")
                print(f"[Retrying] Backing off for {delay} seconds...")
                time.sleep(delay)
                delay *= 2

    def _verify_and_cache_paths(self):
        print(f"[Financial Pentad System] Inspecting Hugging Face Hub Repositories (Seed: {self.inference_seed})...")
        self.model_paths = {}

        for role, config in self.model_configs.items():
            repo_id = config["repo_id"]

            files = list_repo_files(repo_id=repo_id)
            gguf_files = [f for f in files if f.endswith('.gguf')]
            if not gguf_files:
                raise FileNotFoundError(f"No .gguf files found in repository {repo_id}")

            filename = config.get("filename")
            if not filename or filename not in gguf_files:
                filename = gguf_files[0]

            path = self._robust_download(repo_id, filename)

            if not os.path.exists(path) or os.path.getsize(path) < 1024:
                raise FileNotFoundError(f"Model file for {role} at {path} is invalid or missing.")

            self.model_paths[role] = path
            print(f"[Verified] {role} -> {path}\n")

    def _load_agent(self, role: str) -> Llama:
        if role in self.agents:
            return self.agents[role]

        path = self.model_paths[role]
        print(f"[Load] Initializing {role} into execution memory...")

        # Exact hardware allocation: One 100% CPU, the rest Mixed/Hybrid Offload
        if role == "triage":
            n_gpu = 0   # 100% CPU Execution
            n_ctx = 4096
        elif role == "derivatives":
            n_gpu = 35  # Mixed hybrid offload
            n_ctx = 4096
        elif role == "macro":
            n_gpu = 20  # Mixed hybrid offload
            n_ctx = 4096
        elif role == "liquidity":
            n_gpu = 24  # Mixed hybrid offload
            n_ctx = 4096
        elif role == "compliance_reasoner":
            n_gpu = 35  # Mixed hybrid offload for 70B
            n_ctx = 8192
        else:
            n_gpu = 20
            n_ctx = 4096

        agent = Llama(
            model_path=path,
            n_gpu_layers=n_gpu,
            n_ctx=n_ctx,
            n_threads=8,
            use_mmap=True,
            seed=self.inference_seed,
            verbose=False
        )
        self.agents[role] = agent
        return agent

    def _unload_agent(self, role: str):
        if role in self.agents:
            del self.agents[role]
            gc.collect()

    def _capture_nvidia_telemetry(self) -> dict:
        telemetry = {"status": "unavailable"}
        if not shutil.which("nvidia-smi"):
            return telemetry
        try:
            cmd = [
                "nvidia-smi",
                "--query-gpu=name,temperature.gpu,utilization.gpu,memory.used,memory.total,power.draw",
                "--format=csv,noheader,nounits"
            ]
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            parts = [p.strip() for p in result.stdout.strip().split(",")]
            if len(parts) >= 6:
                telemetry = {
                    "gpu_name": parts[0],
                    "temp_c": parts[1],
                    "utilization_pct": parts[2],
                    "memory_used_mib": parts[3],
                    "memory_total_mib": parts[4],
                    "power_draw_w": parts[5],
                    "status": "active"
                }
        except Exception as e:
            telemetry["error"] = str(e)
        return telemetry

    def evaluate_portfolio_pentad(self, portfolio_data: str) -> dict:
        # Stage 1: Triage Anomaly Screening (100% CPU Agent)
        triage_model = self._load_agent("triage")
        triage_res = triage_model(
            prompt=f"<|User|>Analyze transaction telemetry patterns and identify liquidity anomaly vectors: {portfolio_data}<|Assistant|>",
            max_tokens=256, temperature=0.1, seed=self.inference_seed
        )
        triage_text = triage_res["choices"][0]["text"].strip()
        self._unload_agent("triage")

        # Stage 2: Derivatives Risk Analysis (Mixed Hybrid Offload)
        deriv_model = self._load_agent("derivatives")
        deriv_res = deriv_model(
            prompt=f"<|User|>Evaluate cross-currency volatility and structural risk indicators based on findings: {triage_text}<|Assistant|>",
            max_tokens=256, temperature=0.1, seed=self.inference_seed
        )
        deriv_text = deriv_res["choices"][0]["text"].strip()
        self._unload_agent("derivatives")

        # Stage 3: Macroeconomic Strategy (Mixed Hybrid Offload)
        macro_model = self._load_agent("macro")
        macro_res = macro_model(
            prompt=f"<|User|>Assess sovereign debt exposure metrics from derivatives report: {deriv_text}<|Assistant|>",
            max_tokens=256, temperature=0.1, seed=self.inference_seed
        )
        macro_text = macro_res["choices"][0]["text"].strip()
        self._unload_agent("macro")

        # Stage 4: Liquidity Audit (Mixed Hybrid Offload)
        liq_model = self._load_agent("liquidity")
        liq_res = liq_model(
            prompt=f"<|User|>Verify counterparty reserve thresholds given macro assessment: {macro_text}<|Assistant|>",
            max_tokens=256, temperature=0.1, seed=self.inference_seed
        )
        liq_text = liq_res["choices"][0]["text"].strip()
        self._unload_agent("liquidity")

        # Stage 5: Chief Analytical Officer Synthesis (70B Mixed Hybrid Offload - Reframed for Academic Risk Modeling)
        compliance_model = self._load_agent("compliance_reasoner")
        compliance_res = compliance_model(
            prompt=f"<|User|>Synthesize multi-agent financial risk assessment:\n- Triage: {triage_text}\n- Derivatives: {deriv_text}\n- Macro: {macro_text}\n- Liquidity: {liq_text}\n\nGenerate a comprehensive structural risk telemetry report and technical variance mitigation protocol:<|Assistant|>",
            max_tokens=1024, temperature=0.2, top_p=0.95, seed=self.inference_seed
        )

        report_text = compliance_res["choices"][0]["text"].strip()
        report_hash = hashlib.sha256(report_text.encode('utf-8')).hexdigest()
        hw_metrics = self._capture_nvidia_telemetry()
        self._unload_agent("compliance_reasoner")

        return {
            "compliance_report": report_text,
            "report_hash": report_hash,
            "hardware_telemetry": hw_metrics
        }

if __name__ == "__main__":
    model_configs = {
        "triage": {
            "repo_id": "frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf"
        },
        "derivatives": {
            "repo_id": "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
        },
        "macro": {
            "repo_id": "frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf"
        },
        "liquidity": {
            "repo_id": "frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2-gguf"
        },
        "compliance_reasoner": {
            "repo_id": "frankmorales2020/llama-3.1-70b-topo-governed-gguf"
        }
    }

    cluster = FinancialPentadCluster(model_configs=model_configs, inference_seed=123)
    sample_portfolio = "Sudden liquidity contraction in emerging market sovereign debt holdings alongside anomalous volatility spikes in cross-currency derivatives post-rate decision."

    print("\n--- RUN 1: PENTAD PORTFOLIO AUDIT ---")
    run_1 = cluster.evaluate_portfolio_pentad(sample_portfolio)
    print(run_1["compliance_report"])

    print("\n--- RUN 2: DETERMINISTIC VERIFICATION RUN ---")
    run_2 = cluster.evaluate_portfolio_pentad(sample_portfolio)
    print(run_2["compliance_report"])

    print("\n--- BITWISE CONSISTENCY & AUDIT ---")
    match_status = run_1["report_hash"] == run_2["report_hash"]
    telemetry = run_1["hardware_telemetry"]

    if telemetry.get("status") == "active":
        print(f"GPU Accelerator : {telemetry['gpu_name']}")
        print(f"VRAM Allocation : {telemetry['memory_used_mib']} MiB / {telemetry['memory_total_mib']} MiB")
        print(f"GPU Core Temp   : {telemetry['temp_c']} °C")
        print(f"Power Draw      : {telemetry['power_draw_w']} W")
        print(f"Inference Seed  : {cluster.inference_seed}")
        print(f"Run 1 SHA-256   : {run_1['report_hash']}")
        print(f"Run 2 SHA-256   : {run_2['report_hash']}")
        print(f"Bitwise Match   : {'PASS (Identical Output Verified)' if match_status else 'FAIL (Variance Detected)'}")

    assert match_status, "Assertion Failed: Output variance detected across pipeline execution despite locked seed 123."

[Financial Pentad System] Inspecting Hugging Face Hub Repositories (Seed: 123)...
[Hugging Face Hub] Downloading 'muse-glimmer-30b-card-faithful-Q4_K_M.gguf' from frankmorales2020/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf (Attempt 1/5)...
[Verified] triage -> /root/.cache/huggingface/hub/models--frankmorales2020--muse-glimmer-30b-topo-governed-cbp-fineweb-gguf/snapshots/4c01f61a847c4cbdac740852f56f85f0c1855f5b/muse-glimmer-30b-card-faithful-Q4_K_M.gguf

[Hugging Face Hub] Downloading 'mixtral_8x22b_q4_k_m.gguf' from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF (Attempt 1/5)...
[Verified] derivatives -> /root/.cache/huggingface/hub/models--frankmorales2020--mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF/snapshots/33cb918327fe0a9a3e2aec1701ad6f9dae2fc7f9/mixtral_8x22b_q4_k_m.gguf

[Hugging Face Hub] Downloading 'nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf' from frankmorales2020/nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf (Attempt 1/5)...
[Verified] macro -> /root/.cac

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


[Load] Initializing derivatives into execution memory...
[Load] Initializing macro into execution memory...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


[Load] Initializing liquidity into execution memory...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


... (3 outputs left out)

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


[Load] Initializing liquidity into execution memory...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


[Load] Initializing compliance_reasoner into execution memory...
Here is a comprehensive structural risk telemetry report and technical variance mitigation protocol:

**Structural Risk Telemetry Report**

**Executive Summary:**

This report provides a comprehensive analysis of the structural risks associated with sovereign debt exposure in emerging markets. The analysis is based on a combination of quantitative and qualitative metrics, including debt-to-GDP ratios, interest coverage ratios, and structural determinants such as demographics, institutional quality, and commodity dependence.

**Key Findings:**

1. **Debt Sustainability:** The debt-to-GDP ratio for emerging markets has increased significantly over the past decade, with some countries exceeding 60% of GDP.
2. **Interest Burden:** The interest coverage ratio for emerging markets has deteriorated, with some countries facing significant challenges in meeting their interest payments.
3. **Structural Vulnerabilities:** Demographi